In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "Dataset/Missing_Train/processed_train_missing_30.csv"
)

FEATURES = [
    "Setting_1",
    "Setting_2",
    "Sensor_2",
    "Sensor_3",
    "Sensor_4",
    "Sensor_6",
    "Sensor_7",
    "Sensor_8",
    "Sensor_9",
    "Sensor_11",
    "Sensor_12",
    "Sensor_13",
    "Sensor_14",
    "Sensor_15",
    "Sensor_17",
    "Sensor_20",
    "Sensor_21"
]


def create_masked_dataset(df, features):

    data = df.copy()

    mask_columns = []

    for feature in features:

        mask_name = f"{feature}_mask"

        # 1 = observed, 0 = missing
        data[mask_name] = (
            data[feature]
            .notna()
            .astype("float32")
        )

        mask_columns.append(mask_name)

    # Replace missing numerical values with neutral placeholder
    data[features] = (
        data[features]
        .fillna(0.0)
        .astype("float32")
    )

    return data, mask_columns


masked_df, MASK_FEATURES = create_masked_dataset(
    df,
    FEATURES
)

MODEL_FEATURES = FEATURES + MASK_FEATURES

print("Original input features:", len(FEATURES))
print("Mask features:", len(MASK_FEATURES))
print("Total model features:", len(MODEL_FEATURES))

print(
    "NaNs remaining:",
    masked_df[MODEL_FEATURES]
    .isna()
    .sum()
    .sum()
)

Original input features: 17
Mask features: 17
Total model features: 34
NaNs remaining: 0


In [2]:
masked_df[
        [
            "Sensor_2",
            "Sensor_2_mask",
            "Sensor_7",
            "Sensor_7_mask",
            "Sensor_20",
            "Sensor_20_mask"
        ]
    ].head(15)

,Sensor_2,Sensor_2_mask,Sensor_7,Sensor_7_mask,Sensor_20,Sensor_20_mask
0,0.183735,1.0,0.726248,1.0,0.000000,0.0
1,0.283133,1.0,0.628019,1.0,0.666667,1.0
2,0.000000,0.0,0.710145,1.0,0.000000,0.0
3,0.343374,1.0,0.740741,1.0,0.573643,1.0
4,0.000000,0.0,0.000000,0.0,0.000000,0.0
5,0.268072,1.0,0.776167,1.0,0.651163,1.0
6,0.382530,1.0,0.000000,0.0,0.744186,1.0
7,0.406626,1.0,0.644122,1.0,0.643411,1.0
8,0.274096,1.0,0.618357,1.0,0.705426,1.0
9,0.150602,1.0,0.602254,1.0,0.627907,1.0


Implementation

In [3]:
import pandas as pd
import numpy as np

train_missing = pd.read_csv(
    "Dataset/Missing_Train/processed_train_missing_30.csv"
)

test_missing = pd.read_csv(
    "Dataset/Missing_Test/processed_test_missing_30.csv"
)

In [4]:
FEATURES = [
    "Setting_1",
    "Setting_2",
    "Sensor_2",
    "Sensor_3",
    "Sensor_4",
    "Sensor_6",
    "Sensor_7",
    "Sensor_8",
    "Sensor_9",
    "Sensor_11",
    "Sensor_12",
    "Sensor_13",
    "Sensor_14",
    "Sensor_15",
    "Sensor_17",
    "Sensor_20",
    "Sensor_21"
]

TARGET = "RUL"
WINDOW = 30

In [5]:
def create_masked_dataset(df, features):

    data = df.copy()

    mask_features = []

    for feature in features:

        mask_name = feature + "_mask"

        # 1 = observed
        # 0 = missing
        data[mask_name] = (
            data[feature]
            .notna()
            .astype("float32")
        )

        mask_features.append(mask_name)

    # Placeholder only.
    # Mask tells model whether 0 is real or missing.
    data[features] = (
        data[features]
        .fillna(0.0)
        .astype("float32")
    )

    return data, mask_features

In [6]:
train_masked, MASK_FEATURES = create_masked_dataset(
    train_missing,
    FEATURES
)

test_masked, _ = create_masked_dataset(
    test_missing,
    FEATURES
)

MODEL_FEATURES = FEATURES + MASK_FEATURES

print("Input features:", len(MODEL_FEATURES))
# 34

Input features: 34


In [7]:
print(
    train_masked[MODEL_FEATURES]
    .isna()
    .sum()
    .sum()
)

print(
    test_masked[MODEL_FEATURES]
    .isna()
    .sum()
    .sum()
)

0
0


In [8]:
clean_train = pd.read_csv(
    "Dataset/processed_train.csv"
)

np.random.seed(42)

engine_ids = clean_train[
    "Engine_ID"
].unique().copy()

np.random.shuffle(engine_ids)

train_ids = engine_ids[:70]
val_ids = engine_ids[70:85]

In [9]:
train_df_30 = train_masked[
    train_masked["Engine_ID"].isin(train_ids)
].copy()

val_df_30 = train_masked[
    train_masked["Engine_ID"].isin(val_ids)
].copy()

In [10]:
def create_sequences(
    data,
    features,
    target,
    window=30
):

    X = []
    y = []

    for engine_id, engine_data in data.groupby("Engine_ID"):

        engine_data = engine_data.sort_values("Cycle")

        values = (
            engine_data[features]
            .values
            .astype("float32")
        )

        targets = (
            engine_data[target]
            .values
            .astype("float32")
        )

        for i in range(window, len(engine_data)):

            X.append(
                values[i-window:i]
            )

            y.append(
                targets[i]
            )

    return np.array(X), np.array(y)

In [11]:
X_train_30, y_train_30 = create_sequences(
    train_df_30,
    MODEL_FEATURES,
    TARGET,
    WINDOW
)

X_val_30, y_val_30 = create_sequences(
    val_df_30,
    MODEL_FEATURES,
    TARGET,
    WINDOW
)

print(X_train_30.shape)
print(X_val_30.shape)

(12216, 30, 34)
(2720, 30, 34)


BiLSTM

In [12]:
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Bidirectional,
    Dropout,
    Dense
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

tf.keras.backend.clear_session()
tf.random.set_seed(42)

bilstm_30 = Sequential([

    Input(
        shape=(
            WINDOW,
            len(MODEL_FEATURES)
        )
    ),

    Bidirectional(
        LSTM(
            64,
            return_sequences=True
        )
    ),

    Dropout(0.2),

    Bidirectional(
        LSTM(32)
    ),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dense(1)
])

bilstm_30.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

C:\Users\Shivam\AppData\Roaming\Python\Python310\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.2) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [13]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

history = bilstm_30.fit(
    X_train_30,
    y_train_30,

    validation_data=(
        X_val_30,
        y_val_30
    ),

    epochs=100,
    batch_size=64,
    shuffle=False,

    callbacks=[
        early_stop,
        reduce_lr
    ],

    verbose=1
)

Epoch 1/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 13s 30ms/step - loss: 4240.2412 - mae: 54.9535 - val_loss: 1826.2754 - val_mae: 38.4202 - learning_rate: 0.0010
Epoch 2/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - loss: 1783.2399 - mae: 37.5237 - val_loss: 1748.6400 - val_mae: 37.1364 - learning_rate: 0.0010
Epoch 3/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - loss: 1775.5421 - mae: 37.3158 - val_loss: 1748.6272 - val_mae: 37.1340 - learning_rate: 0.0010
Epoch 4/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - loss: 1780.0347 - mae: 37.3366 - val_loss: 1748.5623 - val_mae: 37.1418 - learning_rate: 0.0010
Epoch 5/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - loss: 1775.0979 - mae: 37.3159 - val_loss: 1748.5277 - val_mae: 37.1325 - learning_rate: 0.0010
Epoch 6/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - loss: 1773.4696 - mae: 37.3292 - val_loss: 1747.7358 - val_mae: 37.1043 - learning_rate: 0.0010
Epoch 7/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - loss: 1608.9067 - mae: 35.0615 

In [14]:
def create_final_test_sequences(
    test_df,
    features,
    window=30
):

    X = []
    engine_ids = []

    for engine_id, engine_data in test_df.groupby(
        "Engine_ID"
    ):

        engine_data = engine_data.sort_values(
            "Cycle"
        )

        values = (
            engine_data[features]
            .values
            .astype("float32")
        )

        if len(values) >= window:

            X.append(
                values[-window:]
            )

            engine_ids.append(
                engine_id
            )

    return (
        np.array(X),
        np.array(engine_ids)
    )

In [15]:
X_test_30, test_engine_ids = (
    create_final_test_sequences(
        test_masked,
        MODEL_FEATURES,
        WINDOW
    )
)

print(X_test_30.shape)

(100, 30, 34)


In [16]:
bilstm_pred_30 = bilstm_30.predict(
    X_test_30,
    verbose=0
).ravel()

In [17]:
RUL_CAP = 125

rul_df = pd.read_csv(
    "Dataset/RUL_FD001.csv"
)

if "Engine_ID" not in rul_df.columns:

    rul_df["Engine_ID"] = np.arange(
        1,
        len(rul_df) + 1
    )

rul_df["RUL"] = rul_df["RUL"].clip(upper=RUL_CAP)


In [18]:
prediction_df = pd.DataFrame({
    "Engine_ID": test_engine_ids,
    "Predicted_RUL": bilstm_pred_30
})

comparison = prediction_df.merge(
    rul_df,
    on="Engine_ID"
)

comparison.head()

,Engine_ID,Predicted_RUL,RUL
0,1,118.699760,112
1,2,119.660362,98
2,3,37.446648,69
3,4,85.733650,82
4,5,110.517021,91


In [19]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

def nasa_score(actual, predicted):
    """NASA / PHM08 asymmetric RUL scoring function (Saxena et al., 2008).
    Penalizes late predictions (predicted RUL > actual RUL) more heavily
    than early ones, since a late prediction means maintenance would be
    scheduled after the engine could already have failed.
    """
    d = np.asarray(predicted) - np.asarray(actual)
    s = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)
    return np.sum(s)

actual = comparison["RUL"].values
predicted = comparison["Predicted_RUL"].values

mae = mean_absolute_error(
    actual,
    predicted
)

rmse = np.sqrt(
    mean_squared_error(
        actual,
        predicted
    )
)

r2 = r2_score(
    actual,
    predicted
)

score = nasa_score(
    actual,
    predicted
)

print("30% Missing - BiLSTM")
print("MAE  :", mae)
print("RMSE :", rmse)
print("R\u00b2   :", r2)
print("Score:", score)


30% Missing - BiLSTM
MAE  : 10.87377643585205
RMSE : 14.375072180525303
R²   : 0.8713201880455017
Score: 316.68224874998754


In [20]:
def evaluate(name, actual, predicted):
    predicted = np.asarray(predicted).ravel()
    actual = np.asarray(actual).ravel()

    return {
        "Model": name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted),
        "Score": nasa_score(actual, predicted)
    }

results_30 = []

# Add your existing BiLSTM result
results_30.append(
    evaluate(
        "BiLSTM",
        actual,
        bilstm_pred_30
    )
)


LSTM

In [21]:
from sklearn.preprocessing import StandardScaler

y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(
    y_train_30.reshape(-1, 1)
).ravel()

y_val_scaled = y_scaler.transform(
    y_val_30.reshape(-1, 1)
).ravel()

In [22]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)

lstm_30 = Sequential([
    Input(
        shape=(WINDOW, len(MODEL_FEATURES))
    ),

    LSTM(
        64,
        return_sequences=True
    ),

    Dropout(0.2),

    LSTM(32),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dense(1)
])

lstm_30.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

In [23]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

history_lstm_30 = lstm_30.fit(
    X_train_30,
    y_train_scaled,

    validation_data=(
        X_val_30,
        y_val_scaled
    ),

    epochs=100,
    batch_size=64,
    shuffle=False,

    callbacks=[
        early_stop,
        reduce_lr
    ],

    verbose=1
)

Epoch 1/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - loss: 0.8102 - mae: 0.7678 - val_loss: 0.8643 - val_mae: 0.8233 - learning_rate: 0.0010
Epoch 2/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - loss: 0.6306 - mae: 0.6730 - val_loss: 0.3362 - val_mae: 0.4837 - learning_rate: 0.0010
Epoch 3/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - loss: 0.4246 - mae: 0.5320 - val_loss: 0.3197 - val_mae: 0.4839 - learning_rate: 0.0010
Epoch 4/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - loss: 0.3827 - mae: 0.5032 - val_loss: 0.2999 - val_mae: 0.4664 - learning_rate: 0.0010
Epoch 5/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - loss: 0.3616 - mae: 0.4846 - val_loss: 0.2786 - val_mae: 0.4466 - learning_rate: 0.0010
Epoch 6/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 0.3354 - mae: 0.4655 - val_loss: 0.2483 - val_mae: 0.4132 - learning_rate: 0.0010
Epoch 7/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 0.3223 - mae: 0.4540 - val_loss: 0.2439 - val_mae: 0.4114 - learning_rate:

In [24]:
lstm_pred_scaled = lstm_30.predict(
    X_test_30,
    verbose=0
)

lstm_pred_30 = y_scaler.inverse_transform(
    lstm_pred_scaled
).ravel()

In [25]:
print(
    evaluate(
        "LSTM",
        actual,
        lstm_pred_30
    )
)

results_30.append(
    evaluate(
        "LSTM",
        actual,
        lstm_pred_30
    )
)

{'Model': 'LSTM', 'MAE': 12.013362884521484, 'RMSE': np.float64(15.832626380530451), 'R2': 0.8439024090766907, 'Score': np.float64(471.9040331535589)}


GRU

In [26]:
from tensorflow.keras.layers import GRU

tf.keras.backend.clear_session()
tf.random.set_seed(42)

gru_30 = Sequential([
    Input(
        shape=(WINDOW, len(MODEL_FEATURES))
    ),

    GRU(
        64,
        return_sequences=True
    ),

    Dropout(0.2),

    GRU(32),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dense(1)
])

gru_30.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss="mse",
    metrics=["mae"]
)

gru_30.fit(
    X_train_30,
    y_train_30,
    validation_data=(X_val_30, y_val_30),
    epochs=100,
    batch_size=64,
    shuffle=False,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )
    ],
    verbose=1
)

gru_pred_30 = gru_30.predict(
    X_test_30,
    verbose=0
).ravel()

results_30.append(
    evaluate(
        "GRU",
        actual,
        gru_pred_30
    )
)

Epoch 1/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - loss: 5327.0151 - mae: 62.2928 - val_loss: 2941.2727 - val_mae: 47.0029
Epoch 2/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - loss: 2060.4351 - mae: 40.0323 - val_loss: 1762.9272 - val_mae: 37.5953
Epoch 3/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - loss: 1791.9124 - mae: 37.4821 - val_loss: 1749.4823 - val_mae: 37.2538
Epoch 4/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - loss: 1802.3787 - mae: 37.5096 - val_loss: 1749.2587 - val_mae: 37.2406
Epoch 5/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - loss: 1800.8058 - mae: 37.5023 - val_loss: 1749.0579 - val_mae: 37.2268
Epoch 6/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - loss: 1786.2943 - mae: 37.3629 - val_loss: 1749.1620 - val_mae: 37.2343
Epoch 7/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - loss: 1804.9413 - mae: 37.5081 - val_loss: 1749.1908 - val_mae: 37.2365
Epoch 8/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - loss: 1804.5629 - mae: 37.5629 - val_loss: 1749.

TCN

In [27]:
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    Dropout,
    Add,
    Activation,
    Lambda,
    Dense
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

def tcn_block(
    x,
    filters,
    kernel_size,
    dilation_rate,
    dropout=0.1
):
    residual = x

    x = Conv1D(
        filters,
        kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        activation="relu"
    )(x)

    x = Dropout(dropout)(x)

    x = Conv1D(
        filters,
        kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        activation="relu"
    )(x)

    x = Dropout(dropout)(x)

    if residual.shape[-1] != filters:
        residual = Conv1D(
            filters,
            1,
            padding="same"
        )(residual)

    x = Add()([x, residual])

    return Activation("relu")(x)

In [28]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

inputs = Input(
    shape=(WINDOW, len(MODEL_FEATURES))
)

x = tcn_block(inputs, 32, 2, 1)
x = tcn_block(x, 32, 2, 2)
x = tcn_block(x, 32, 2, 4)
x = tcn_block(x, 32, 2, 8)

x = Lambda(
    lambda z: z[:, -1, :]
)(x)

x = Dense(
    32,
    activation="relu"
)(x)

x = Dropout(0.1)(x)

outputs = Dense(1)(x)

tcn_30 = Model(
    inputs,
    outputs
)

tcn_30.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss="mse",
    metrics=["mae"]
)

tcn_30.fit(
    X_train_30,
    y_train_30,
    validation_data=(X_val_30, y_val_30),
    epochs=100,
    batch_size=64,
    shuffle=True,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=15,
            restore_best_weights=True
        )
    ],
    verbose=1
)

tcn_pred_30 = tcn_30.predict(
    X_test_30,
    verbose=0
).ravel()

results_30.append(
    evaluate(
        "TCN",
        actual,
        tcn_pred_30
    )
)

Epoch 1/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - loss: 2066.5918 - mae: 36.3195 - val_loss: 602.2119 - val_mae: 19.9107
Epoch 2/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 752.4671 - mae: 22.2689 - val_loss: 521.3002 - val_mae: 17.9356
Epoch 3/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 651.1191 - mae: 20.1904 - val_loss: 522.4749 - val_mae: 18.4694
Epoch 4/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 586.8237 - mae: 19.1034 - val_loss: 458.9390 - val_mae: 17.3503
Epoch 5/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 564.3032 - mae: 18.6941 - val_loss: 540.5604 - val_mae: 19.2944
Epoch 6/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 531.2608 - mae: 18.1629 - val_loss: 441.5248 - val_mae: 17.2448
Epoch 7/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 495.4135 - mae: 17.3968 - val_loss: 473.3698 - val_mae: 18.1458
Epoch 8/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 467.5157 - mae: 16.8300 - val_loss: 423.0368 - val_mae:

Random Forest

In [29]:
X_train_flat_30 = X_train_30.reshape(
    X_train_30.shape[0],
    -1
)

X_test_flat_30 = X_test_30.reshape(
    X_test_30.shape[0],
    -1
)

In [30]:
from sklearn.ensemble import RandomForestRegressor

rf_30 = RandomForestRegressor(
    n_estimators=500,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf_30.fit(
    X_train_flat_30,
    y_train_30
)

rf_pred_30 = rf_30.predict(
    X_test_flat_30
)

results_30.append(
    evaluate(
        "Random Forest",
        actual,
        rf_pred_30
    )
)

XGBoost

In [31]:
from xgboost import XGBRegressor

X_val_flat_30 = X_val_30.reshape(
    X_val_30.shape[0],
    -1
)

xgb_30 = XGBRegressor(
    n_estimators=700,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_30.fit(
    X_train_flat_30,
    y_train_30,
    eval_set=[
        (
            X_val_flat_30,
            y_val_30
        )
    ],
    verbose=False
)

xgb_pred_30 = xgb_30.predict(
    X_test_flat_30
)

results_30.append(
    evaluate(
        "XGBoost",
        actual,
        xgb_pred_30
    )
)

In [32]:
results_30_df = pd.DataFrame(
    results_30
).sort_values("RMSE")

print(results_30_df)

           Model        MAE       RMSE        R2        Score
0         BiLSTM  10.873776  14.375072  0.871320   316.682249
5        XGBoost  11.779650  15.286254  0.854490   426.770050
2            GRU  12.102745  15.461114  0.851142   425.392101
1           LSTM  12.013363  15.832626  0.843902   471.904033
4  Random Forest  14.459540  19.091872  0.773020  1108.767506
3            TCN  16.187000  21.887245  0.701687  3680.277767
